# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")
openai = OpenAI(api_key=api_key)


OPENAI_API_KEY environment variable is set.


In [4]:
# here is the question; type over this to ask something new
system_prompt = """ You are an expert technical assistant. 
Answer technical questions accurately and clearly. 
Explain complex concepts in simple terms and provide practical examples or code when useful."""

user_prompt = """
Answer the following technical question:
What is a message queue in backend development?
Please explain the answer in simple terms first, then provide a more technical explanation if necessary. Include a practical example or code example where useful.
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]
def get_gpt_answer():
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True
    )

    # Display the answer in real-time
    display_handle = display(Markdown("**Answer:**"), display_id=True)
    answer_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content:
            answer_text += delta.content
            update_display(Markdown(f"**Answer:**\n\n{answer_text}"), display_id=display_handle.display_id)


In [8]:
get_gpt_answer()

**Answer:**

### Simple Explanation of Message Queue

A message queue is like a postal service for computer applications. Imagine you have different teams in a company that need to communicate with each other. Instead of talking directly, they send letters (messages) to a mailbox (the queue), and each team can read the letters at their own pace. 

In backend development, a message queue helps different parts of an application (or different applications) communicate without being tightly connected. This means that one part can send a message to the queue and continue doing its job without waiting for the other part to respond right away. The receiving part can check the queue at a later time, read the message, and process it when it's ready.

### Technical Explanation of Message Queue

In more technical terms, a message queue is an asynchronous communication mechanism that allows different components of a system to send messages back and forth. It decouples the sender (producer) and the receiver (consumer) of the messages, allowing them to operate independently.

- **Producers** are components or services that send messages to the queue.
- **Consumers** are the services/components that read and process the messages from the queue.
- The messages can include data such as requests to perform actions, status updates, or any other information needed for processing.

Message queues often provide features like message durability (ensuring messages are not lost), ordering (ensuring messages are processed in the order they were received), and error handling (managing failures in processing).

### Practical Example

Let's consider a simple example using RabbitMQ, a popular message queue system. Here’s a Python example using the `pika` library to send and receive messages:

1. **Install `pika`**:
   ```bash
   pip install pika
   ```

2. **Producer Code** (sends messages to the queue):
   ```python
   import pika

   # Establish a connection to the RabbitMQ server
   connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
   channel = connection.channel()

   # Declare a queue named 'task_queue'
   channel.queue_declare(queue='task_queue', durable=True)

   # Send messages to the queue
   for i in range(5):
       message = f"Task {i}"
       channel.basic_publish(
           exchange='',
           routing_key='task_queue',
           body=message,
           properties=pika.BasicProperties(
               delivery_mode=2,  # Make message persistent
           ))
       print(f" [x] Sent {message}")

   # Close the connection
   connection.close()
   ```

3. **Consumer Code** (receives messages from the queue):
   ```python
   import pika
   import time

   def callback(ch, method, properties, body):
       print(f" [x] Received {body.decode()}")
       time.sleep(1)  # Simulate work
       print(" [x] Done")
       ch.basic_ack(delivery_tag=method.delivery_tag)

   # Establish a connection to the RabbitMQ server
   connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
   channel = connection.channel()

   # Declare the same queue
   channel.queue_declare(queue='task_queue', durable=True)

   # Tell RabbitMQ that this callback function will process messages from the queue
   channel.basic_qos(prefetch_count=1)  # Fair dispatch
   channel.basic_consume(queue='task_queue', on_message_callback=callback)

   print(' [*] Waiting for messages. To exit press CTRL+C')
   channel.start_consuming()
   ```

### Explanation of the Example
- In the producer code, we connect to RabbitMQ, declare a queue named `task_queue`, and send five messages labeled as "Task 0" to "Task 4". The messages are marked as persistent so they are not lost if RabbitMQ crashes.
- In the consumer code, we set up a listener on the same queue that waits for messages. When a message is received, it simulates processing by sleeping for one second, then acknowledges that it has been processed.

By using a message queue, the producer can send messages without worrying about whether the consumer is ready to process them immediately. This arrangement enhances scalability and reliability in backend systems.


In [ ]:
# Get Llama 3.2 to answer